# 4. Expériences ML — Split, modèles, seuil, tuning

Notebook d'expériences ML (séparé du notebook "3 Préparation ML", qui couvre l'analyse de la cible, la chasse aux fuites et la construction de la table Gold) : à partir de la table Gold `workspace.projet_alpha.insurance_claim_fraud_gold`, il réalise :
1. **Split temporel** train/test (80% des sinistres les plus anciens / 20% les plus récents)
2. **Vérifications** : taux de fraude par ensemble, bornes de dates, contrôle des NULL sur les features
3. **Pipeline sklearn** (StandardScaler + OneHotEncoder) et baseline régression logistique
4. **Évaluation** : matrice de confusion, précision/rappel, ROC, feature importance
5. **Comparaison MLflow** : régression logistique, Random Forest, XGBoost
6. **Optimisation du seuil de décision** : courbe PR, contrainte de rappel ≥ 0,90, point mort économique
7. **Tuning** : RandomizedSearchCV + TimeSeriesSplit, seuil calibré sur le dernier fold, enregistrement dans le Model Registry Unity Catalog

Tout ce qui est exécuté ici relit la table Gold depuis Unity Catalog : aucun effet de bord sur le pipeline ETL quotidien ni sur la table Gold.

In [0]:
# Paramètres du notebook
gold_table = "workspace.projet_alpha.insurance_claim_fraud_gold"
table_gold = spark.table(gold_table)

# Expérience MLflow — sur Databricks, le nom doit être un CHEMIN ABSOLU dans le workspace (à adapter à votre utilisateur)
MLFLOW_EXPERIMENT = "/Users/<votre-user>/XPs MLFlow - Insurance Vehicles Fraud Detection"

## 1) Split temporel des données Gold

- En production, le modèle prédira des sinistres futurs jamais vus pendant son entrainement. Ainsi, on sépare les sinistres en deux datasets :
    - 80% des plus anciens pour le dataset d'entrainement
    - 20% des plus récents pour le dataset de test
- Cela évite que des données antérieures se retrouvent dupliquées dans le train et le test, ce qui impliquerait une simple reconnaissance et non une véritable prédiction du modèle.

In [0]:
# Répartition des claim_date par année -> choix de la date de coupure
claim_date_df = spark.sql(f"""
    SELECT EXTRACT(YEAR FROM claim_date) as year_claim_date,
    MAX(claim_date) as max_claim_date,
    MIN(claim_date) as min_claim_date
    FROM {gold_table}
    GROUP BY EXTRACT(YEAR FROM claim_date)
    ORDER BY EXTRACT(YEAR FROM claim_date)""")

display(claim_date_df)

In [0]:
# Train : 80% des sinistres les plus anciens (PERCENT_RANK sur claim_date)
train_df = spark.sql(f"""
                     WITH dataset_classe AS (SELECT *, PERCENT_RANK() OVER (ORDER BY claim_date ASC) as progression FROM {gold_table})

                    SELECT * 
                    FROM dataset_classe 
                    WHERE progression <= 0.80""")

display(train_df)

In [0]:
# Test : 20% des sinistres les plus récents
test_df = spark.sql(f"""
                     WITH dataset_classe AS (SELECT *, PERCENT_RANK() OVER (ORDER BY claim_date ASC) as progression FROM {gold_table})

                    SELECT *
                    FROM dataset_classe 
                    WHERE progression > 0.80""")

display(test_df)

In [0]:
# Séparation features / cible dans train et test
# (claim_id, claim_date, incident_date exclues : clés et dates de découpage, pas des features)
target_col = 'is_fraud'
features_col = [f.name for f in train_df.schema.fields if f.name not in ['claim_id', 'claim_date', 'incident_date', 'claim_status', 'progression', 'is_fraud']]

print("\nFeatures: ", features_col)
print("\nNb features", len(features_col))
print("\nTarget: ", target_col)

X_train = train_df.select(features_col)
y_train = train_df.select(target_col)

X_test = test_df.select(features_col)
y_test = test_df.select(target_col)

print(f"\nX_train: {X_train.count()} lignes x {len(X_train.columns)} colonnes")
display(X_train.limit(5))

print(f"\ny_train: {y_train.count()} lignes x {len(y_train.columns)} colonnes")
display(y_train.limit(5))

print(f"\nX_test: {X_test.count()} lignes x {len(X_test.columns)} colonnes")
display(X_test.limit(5))

print(f"\ny_test: {y_test.count()} lignes x {len(y_test.columns)} colonnes")
display(y_test.limit(5))

### A) Vérification du taux de fraude

In [0]:
# y_train / y_test en vues temporaires pour les interroger en SQL
y_train.createOrReplaceTempView("y_train_v")
y_test.createOrReplaceTempView("y_test_v")

taux_fraude_y_train = spark.sql(f"""
    SELECT count(CASE WHEN is_fraud = 1 THEN 1 END) AS nb_fraudes,
    count(CASE WHEN is_fraud = 0 THEN 1 END) AS nb_non_fraudes,
    count(*) AS nb_total,
    round(count(CASE WHEN is_fraud = 1 THEN 1 END) / count(*) * 100, 2) AS taux_fraude_pct,
    round(count(CASE WHEN is_fraud = 0 THEN 1 END) / count(*) * 100, 2) AS taux_non_fraude_pct
    FROM y_train_v
""")
display(taux_fraude_y_train)

taux_fraude_y_test = spark.sql(f"""
    SELECT count(CASE WHEN is_fraud = 1 THEN 1 END) AS nb_fraudes,
    count(CASE WHEN is_fraud = 0 THEN 1 END) AS nb_non_fraudes,
    count(*) AS nb_total,
    round(count(CASE WHEN is_fraud = 1 THEN 1 END) / count(*) * 100, 2) AS taux_fraude_pct,
    round(count(CASE WHEN is_fraud = 0 THEN 1 END) / count(*) * 100, 2) AS taux_non_fraude_pct
    FROM y_test_v
""")
display(taux_fraude_y_test)

### B) Vérification des années dans train et test

In [0]:
# Bornes de dates train / test
train_df.createOrReplaceTempView("train_v")
test_df.createOrReplaceTempView("test_v")

print("Vérification des années dans train\n")
display(spark.sql(f"""
                  SELECT 
                  min(claim_date) as min_claim_date,
                  EXTRACT(YEAR FROM min(claim_date)) as min_claim_year,
                  max(claim_date) as max_claim_date,
                  EXTRACT(YEAR FROM max(claim_date)) as max_claim_year,
                  min(incident_date) as min_incident_date,
                  EXTRACT(YEAR FROM min(incident_date)) as min_incident_year,
                  max(incident_date) as max_incident_date,
                  EXTRACT(YEAR FROM max(incident_date)) as max_incident_year
                  FROM train_v
                  """))

print("Vérification des années dans test\n")
display(spark.sql(f"""
                  SELECT 
                  min(claim_date) as min_claim_date,
                  EXTRACT(YEAR FROM min(claim_date)) as min_claim_year,
                  max(claim_date) as max_claim_date,
                  EXTRACT(YEAR FROM max(claim_date)) as max_claim_year,
                  min(incident_date) as min_incident_date,
                  EXTRACT(YEAR FROM min(incident_date)) as min_incident_year,
                  max(incident_date) as max_incident_date,
                  EXTRACT(YEAR FROM max(incident_date)) as max_incident_year
                  FROM test_v
                  """))

### C) Commentaire sur le taux de fraude des ensembles train et test
Le taux de fraude entre le train et le test est stable. Le taux de fraude n'a pas dérivé au fil des années donc le test est comparable au train. Par conséquent, le split temporel des données Gold en 80% anciens et 20% récents n'a pas généré de biais temporel.

- Nombre de features : 29
- Target : is_fraud

- Date de coupure train/test : 2025-01-02/03

- Taille des données d'entrainement :
    - X_train: 80037 lignes x 29 colonnes
    - y_train: 80 037 lignes x 1 colonnes
    - Temporalité claim_date : 2021-01-01 à 2025-01-02

- Taille des données de test:
    - X_test: 19 963 lignes x 29 colonnes
    - y_test: 19 963 lignes x 1 colonnes
    - Temporalité claim_date : 2025-01-03 à 2026-01-10

### D) Contrôle des valeurs NULL sur les features

In [0]:
# Les 29 features
feats = ["age", "credit_score", "years_with_company", "previous_claims", "annual_income",
         "policy_duration_years", "premium_amount", "coverage_amount", "deductible",
         "witnesses", "injuries", "vehicle_age", "mileage", "vehicle_value",
         "repair_cost", "towing_cost", "medical_cost", "rental_vehicle_cost", "claim_amount",
         "accident_severity", "is_suspicious", "is_gps_location_match", "is_delayed_reported",
         "is_night_claim", "has_police_report", "reporting_delay_days", "repair_cost_ratio",
         "claim_amount_ratio", "severe_no_police_report"]

In [0]:
# Contrôle NULL par feature + vue d'ensemble
exprs = ",\n  ".join([f"sum(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS `{c}`" for c in feats])
result = spark.sql(f"SELECT\n  {exprs}\nFROM {gold_table}").toPandas().T
result.columns = ["nb_nulls"]
display(result)

null_expr = " OR ".join([f"{c} IS NULL" for c in feats])
synthese = spark.sql(f"""
    SELECT count(*) AS nb_lignes,
           sum(CASE WHEN ({null_expr}) THEN 1 ELSE 0 END) AS nb_lignes_avec_null
    FROM {gold_table}
""")
display(synthese)

## 2) Pipeline sklearn

X (29 features brutes)
├── accident_severity (1 col) ──> OneHotEncoder ──> 4 colonnes binaires
└── les 28 numériques/flags ──> "passthrough" ──> inchangées
                                    ↓
                         matrice finale : 32 colonnes
                                    ↓
                               XGBoost

In [0]:
# Conversions pandas (les 22 features numériques continues sont standardisées,
# OneHotEncoder sur accident_severity, les indicateurs binaires 0/1 passent tels quels)
X_train_pd = X_train.toPandas()
y_train_pd = y_train.toPandas()

X_test_pd = X_test.toPandas()
y_test_pd = y_test.toPandas()

In [0]:
# ColumnTransformer + Pipeline + baseline Logistic Regression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

# Features numériques (hors des features 0/1)
numeric_features = ["age", "credit_score", "years_with_company", "previous_claims", "annual_income",
                 "policy_duration_years", "premium_amount", "coverage_amount", "deductible",
                 "witnesses", "injuries", "vehicle_age", "mileage", "vehicle_value",
                 "repair_cost", "towing_cost", "medical_cost", "rental_vehicle_cost", "claim_amount",
                 "reporting_delay_days", "repair_cost_ratio", "claim_amount_ratio"]

numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler()) # moyenne 0 et écart-type 1 pour uniformiser les échelles
    ])

# Features catégorielles
categorical_features = ["accident_severity"]
categorical_transformer = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore"))  # ignorer les nouvelles catégories au test
    ])

# Assemblage des branches
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="passthrough",  # garder les indicateurs binaires 0/1 tels quels
)

# Pipeline final : préprocesseur + classifieur
clf = Pipeline(
    steps=[("preprocessor", preprocessor), ("classifier", LogisticRegression())]
)

X_train_transform = preprocessor.fit_transform(X_train_pd)
print(preprocessor.get_feature_names_out())
print(X_train_transform.shape)

# Entrainement sur train
clf.fit(X_train_pd, y_train_pd["is_fraud"])

# Évaluation sur test
score = clf.score(X_test_pd, y_test_pd["is_fraud"])
print("\nAccuracy Logistic Regression:", round(score,2))

# Prédictions
predictions = clf.predict(X_test_pd)
predictions_probas = clf.predict_proba(X_test_pd)

print("\nFraudes predites:", sum(predictions))
print("\nFraudes dans test:", sum(y_test_pd["is_fraud"]))

### A) Premiers résultats avec Logistic Regression

- Accuracy Logistic Regression: 0.89
- Fraudes predites: 2510
- Fraudes dans test: 3992
- Precision: 0.87 (parmi les prédictions, combien sont de vraies fraudes)
- Recall: 0.55 (parmi les vrais fraudes, combien on été prédites)

In [0]:
%pip install seaborn

In [0]:
# Matrice de confusion, ROC, feature importance
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test_pd["is_fraud"], predictions)
print(cm)

from sklearn.metrics import precision_score, recall_score
precision = precision_score(y_test_pd["is_fraud"], predictions)
recall = recall_score(y_test_pd["is_fraud"], predictions)
print("\nPrecision:", round(precision,2))
print("\nRecall:", round(recall,2))

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, cmap="Blues", fmt="d")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# ROC curve : tracée sur la probabilité de la CLASSE POSITIVE uniquement
# (predict_proba renvoie une probabilité par classe ; colonne 1 = "fraude")
from sklearn.metrics import roc_curve, auc
fpr, tpr, thresholds = roc_curve(y_test_pd["is_fraud"], predictions_probas[:, 1])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color="darkorange", lw=2, label="ROC curve (area = %0.2f)" % roc_auc)
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic")
plt.legend(loc="lower right")
plt.show()

# Feature importance (coefficients signés de la régression logistique)
importances = clf.named_steps["classifier"].coef_[0]
feature_importances = pd.DataFrame(
    {"feature": preprocessor.get_feature_names_out(), "importance": importances}
)
feature_importances = feature_importances.sort_values(by="importance", ascending=False)
print(feature_importances)

plt.figure(figsize=(10, 8))
sns.barplot(x="importance", y="feature", data=feature_importances[:10])
plt.title("Feature Importance les + importantes")
plt.show()

plt.figure(figsize=(10, 8))
sns.barplot(x="importance", y="feature", data=feature_importances[20:])
plt.title("Feature Importance les - importantes")
plt.show()

### B) Analyse des résultats de la régression logistique (baseline)

**Contexte.** Premier modèle entraîné sur la table Gold : régression logistique intégrée à un pipeline sklearn (StandardScaler sur les 22 features numériques continues, OneHotEncoder sur accident_severity, les 6 indicateurs binaires 0/1 passés tels quels). Entraînement sur 80 037 sinistres (2021-01-01 → 2025-01-02), évaluation sur 19 963 sinistres plus récents (2025-01-03 → 2026-01-10), jamais vus à l'entraînement (split temporel, taux de fraude stable 20,05% / 20,00%).

**1. Impact de la standardisation.** Sans StandardScaler, le modèle n'atteignait pas la convergence (plafond des 100 itérations du solveur lbfgs) et restait au niveau du classifieur trivial : accuracy 0,80, soit exactement le taux de la classe majoritaire (79,96%). Après standardisation des échelles, la convergence est atteinte et l'accuracy monte à **0,89**. La correction n'était donc pas cosmétique : elle a débloqué l'apprentissage réel du modèle, au-delà de la prédiction systématique "pas de fraude".

**2. Lecture métier de la matrice de confusion.**

| | Prédit honnête | Prédit fraude |
| --- | --- | --- |
| Réellement honnête | 15 653 (VN) | 318 (FP) |
| Réellement frauduleux | 1 800 (FN) | 2 192 (VP) |

- Précision : **0,87** — lorsque le modèle déclenche une alerte, il a raison dans 87% des cas (318 accusations infondées).
- Rappel : **0,55** — seules 55% des fraudes réelles sont détectées (2 192 sur 3 992) ; 1 800 fraudes passent inaperçues.
- L'objectif métier de 90% de rappel n'est pas atteint : le rôle de cette baseline est de quantifier le point de départ, non de l'atteindre.
- Asymétrie des coûts : un faux négatif (fraude ratée, indemnisation indue) coûte bien plus qu'un faux positif (contrôle manuel supplémentaire). La métrique pilote du projet est donc le rappel, la précision étant suivie en parallèle pour borner la charge d'investigation.

**3. Capacité de discrimination (AUC).** L'AUC calculée sur les probabilités prédites vaut **0,86**. Une première version tracée sur les labels 0/1 donnait une courbe dégénérée (aire 0,76) : une courbe ROC doit être construite sur les scores continus, car elle évalue le tri sur l'ensemble des seuils possibles. L'écart entre le rappel de 55% (seuil 0,5) et l'AUC de 0,86 indique que l'information de discrimination est présente mais sous-exploitée par le seuil par défaut : le réglage du seuil de décision constituera un levier majeur, en complément du changement de modèle.

**4. Interprétabilité.** Les coefficients standardisés confirment que le modèle s'appuie sur les facteurs documentés du risque de fraude :

- `is_suspicious` domine très largement (5,32), devant `is_delayed_reported` (1,63), `severe_no_police_report` (1,60) et `is_night_claim` (1,47) — soit exactement les Fraud Indicators disponibles au dépôt, validés lors de la chasse aux fuites ;
- `is_gps_location_match` est fortement négatif (−2,43) : l'absence de correspondance GPS est bien un signal de fraude (49% de fraude en NO) ;
- les features créées (`repair_cost_ratio`, `claim_amount_ratio`, `reporting_delay_days`) pèsent peu dans un modèle linéaire ; leur apport sera réévalué avec un modèle non linéaire.

Nuance de lecture : pour la régression logistique, ces importances sont des **coefficients signés** (direction et magnitude), rendus comparables entre eux par la standardisation — à ne pas confondre avec les importances d'arbres.

**5. Limites identifiées et suite du plan.**

- modèle linéaire : ne capte pas les interactions entre indicateurs (les arbres le feront) ;
- déséquilibre 20/80 non traité (aucune pondération de classes) ;
- seuil de décision figé à 0,5.

Prochaines étapes : XGBoost avec `scale_pos_weight`, suivi et comparaison des expériences sous MLflow, puis optimisation du seuil de décision sur la courbe précision/rappel pour se rapprocher de l'objectif de 90% de rappel.

**Référence de comparaison (baseline à battre)** : Accuracy 0,89 · Précision 0,87 · Rappel 0,55 · AUC 0,86.

## 3) Expériences MLflow — 3 modèles

- Logistic Regression
- XGBoost
- Random Forest

In [0]:
# Fonction générique : entraîne un modèle, l'évalue et trace tout dans MLflow
import mlflow

mlflow.set_experiment(MLFLOW_EXPERIMENT)

def train_insurance_vehicles_fraud_model(X_train_pd, y_train_pd, X_test_pd, y_test_pd, classifier_model, classifier_model_name):
    import mlflow
    import mlflow.sklearn
    from sklearn.pipeline import Pipeline
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import precision_score, recall_score
    from sklearn.metrics import confusion_matrix
    import seaborn as sns
    import matplotlib.pyplot as plt
    import pandas as pd
    from sklearn.metrics import roc_curve, auc
  
    with mlflow.start_run(run_name=classifier_model_name):
        # Features numériques (hors des features 0/1)
        numeric_features = ["age", "credit_score", "years_with_company", "previous_claims","annual_income","policy_duration_years", "premium_amount", "coverage_amount", "deductible",
        "witnesses", "injuries", "vehicle_age", "mileage", "vehicle_value",
        "repair_cost", "towing_cost", "medical_cost", "rental_vehicle_cost", "claim_amount","reporting_delay_days", "repair_cost_ratio", "claim_amount_ratio"]

        numeric_transformer = Pipeline(
            steps=[
                ("scaler", StandardScaler())
            ])

        # Features catégorielles
        categorical_features = ["accident_severity"]
        categorical_transformer = Pipeline(
            steps=[
                ("encoder", OneHotEncoder(handle_unknown="ignore"))  # ignorer les nouvelles catégories au test
            ]
        )

        # Assemblage des branches avec ColumnTransformer
        preprocessor = ColumnTransformer(
            transformers=[
                ("num", numeric_transformer, numeric_features),
                ("cat", categorical_transformer, categorical_features),
            ],
            remainder="passthrough",  # garder les autres colonnes
        )

        # Pipeline final
        pipeline = Pipeline(
            steps=[("preprocessor", preprocessor), ("classifier", classifier_model)])
   
        # Hyperparamètres : modèle + Pipeline
        print (f"\n==Training model : {classifier_model_name}==")
        mlflow.set_tag("classifier_model_name", classifier_model_name)
        mlflow.log_params(classifier_model.get_params())

        model = pipeline.fit(X_train_pd, y_train_pd["is_fraud"])

        # Prédictions
        predictions = pipeline.predict(X_test_pd)
        predictions_probas = pipeline.predict_proba(X_test_pd)

        print("\nFraudes predites:", sum(predictions))
        print("\nFraudes dans test:", sum(y_test_pd["is_fraud"]))

        # Matrice de confusion
        cm = confusion_matrix(y_test_pd["is_fraud"], predictions)
        print(f"Matrice de confusion\n", cm)

        confusion_matrix_fig = plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, cmap="Blues", fmt="d")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.title("Confusion Matrix")
        plt.show()

        mlflow.log_figure(confusion_matrix_fig, "confusion_matrix.png")

        # ROC curve
        fpr, tpr, thresholds = roc_curve(y_test_pd["is_fraud"], predictions_probas[:, 1])
        roc_auc = auc(fpr, tpr)

        # L'AUC est LA métrique de comparaison entre les 3 modèles
        mlflow.log_metric("roc_auc", roc_auc)

        roc_curve_fig = plt.figure(figsize=(10, 8))
        plt.plot(fpr, tpr, color="darkorange", lw=2, label="ROC curve (area = %0.2f)" % roc_auc)
        plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("Receiver Operating Characteristic")
        plt.legend(loc="lower right")
        plt.show()

        mlflow.log_figure(roc_curve_fig, "roc_curve.png")

        # Feature importance : les arbres (XGBoost, Random Forest) exposent feature_importances_,
        # la régression logistique expose des coefficients signés (coef_)
        classifier = pipeline.named_steps["classifier"]
        if hasattr(classifier, "feature_importances_"):
            importances = classifier.feature_importances_
        else:
            importances = classifier.coef_[0]
        feature_importances = pd.DataFrame(
            {"feature": preprocessor.get_feature_names_out(), "importance": importances}
        )
        feature_importances = feature_importances.sort_values(by="importance", ascending=False)
        print(feature_importances)

        feat_importance_pos_fig = plt.figure(figsize=(10, 8))
        sns.barplot(x="importance", y="feature", data=feature_importances[:10])
        plt.title("Feature Importance les + importantes")
        plt.show()

        mlflow.log_figure(feat_importance_pos_fig, "features_importance_positive.png")

        feat_importance_neg_fig = plt.figure(figsize=(10, 8))
        sns.barplot(x="importance", y="feature", data=feature_importances[20:])
        plt.title("Feature Importance les - importantes")
        plt.show()

        mlflow.log_figure(feat_importance_neg_fig, "features_importance_negative.png")

        # Métriques d'évaluation sur le test
        metrics = ["accuracy", "recall", "precision"]
        for metric in metrics:
            metricValue=None
            if (metric=="accuracy"):
                score = pipeline.score(X_test_pd, y_test_pd["is_fraud"])
                metricValue = score
            if (metric=="recall"):
                recall = recall_score(y_test_pd["is_fraud"], predictions)
                metricValue = recall
            if (metric=="precision"):
                precision = precision_score(y_test_pd["is_fraud"], predictions)
                metricValue = precision
            if (metricValue!=None):
                print("\n%s: %s" % (metric, round(metricValue,2)))
                mlflow.log_metric(metric, metricValue)
   
        # Le pipeline sklearn complet (préprocesseur + classifieur)
        mlflow.sklearn.log_model(pipeline, "pipeline")
   
        print("\n==Experiment run complete==")

        return pipeline

In [0]:
%pip install xgboost

In [0]:
# Entraîner et tracer les 3 modèles v1
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models_name = [(LogisticRegression(), "logreg_baseline"),
              (RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=42), "random_forest_v1"),
              (XGBClassifier(n_jobs=-1, random_state=42, scale_pos_weight=3.99), "xgboost_v1") ]

for model_instance,model_name in models_name:
    train_insurance_vehicles_fraud_model(X_train_pd, y_train_pd, X_test_pd, y_test_pd, model_instance,  model_name)

### A) Analyse des résultats dans MLflow

 AUC ~0,86 pour les trois familles de modèles → la capacité de discrimination est saturée par la qualité des features ; l'optimisation du seuil de décision devient le levier principal vers l'objectif de rappel 90%.

In [0]:
# Comparer les runs de l'expérience
import mlflow
exp = mlflow.set_experiment(MLFLOW_EXPERIMENT)
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], output_format="pandas")
cols = ["tags.mlflow.runName", "metrics.roc_auc", "metrics.recall", "metrics.precision", "metrics.accuracy"]
print(runs[cols].sort_values("tags.mlflow.runName").to_string(index=False))

#### Étape 1 — Courbe PR + seuil

In [0]:
# Charger les 3 pipelines entraînés depuis MLflow
import mlflow

run_names = ["xgboost_v1", "random_forest_v1", "logreg_baseline"]
models_logged=[]
for run_name in run_names:
    run_id = runs[runs["tags.mlflow.runName"]==run_name]["run_id"].values[0]
    print(run_id)
    artifact_path = "pipeline" # dossier d'artefact défini lors de mlflow.log_model()
    model_uri = f"runs:/{run_id}/{artifact_path}"
    model = mlflow.sklearn.load_model(model_uri)
    models_logged.append(model)
    print(f"Model {run_name} loaded")

models_runs = {run_name: model for run_name, model in zip(run_names, models_logged)}
print(models_runs)

#### Étape 2 — Probabilités de la classe positive

In [0]:
# Probabilités de la classe positive + courbes ROC pour les 3 modèles
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

roc_metrics={}
probas={}
for model_name, model in models_runs.items():
    roc_metrics_models=[]
    print(f"Model {model_name}")
    predictions_probas = model.predict_proba(X_test_pd)
    probas[model_name]=predictions_probas[:, 1]   # colonne 1 = classe positive ("fraude")

    fpr, tpr, thresholds = roc_curve(y_test_pd["is_fraud"], predictions_probas[:, 1])
    roc_auc = auc(fpr, tpr)
    roc_metrics_models.append(roc_auc)
    roc_metrics_models.append(fpr)
    roc_metrics_models.append(tpr)
    roc_metrics_models.append(thresholds)
    roc_metrics[model_name] = roc_metrics_models

#### Étape 3 — Courbe PR

In [0]:
# Courbes précision/rappel des 3 modèles
from sklearn.metrics import precision_recall_curve
precision_recall_curve_metrics={}

for model_name, model in models_runs.items():
    precision_recall_curve_metrics_models=[]
    print(f"Model {model_name}")
    predictions_probas = probas[model_name]
    precision, recall, thresholds = precision_recall_curve(y_test_pd["is_fraud"], predictions_probas)
    precision_recall_curve_metrics_models.append(precision)
    precision_recall_curve_metrics_models.append(recall)
    precision_recall_curve_metrics_models.append(thresholds)
    precision_recall_curve_metrics[model_name] = precision_recall_curve_metrics_models

In [0]:
# Les 3 courbes PR superposées sur UNE figure
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))   # le canevas commun est créé AVANT la boucle

for model_name, metrics in precision_recall_curve_metrics.items():
    # metrics = [precision, recall, thresholds] : rappel en abscisse, précision en ordonnée
    plt.plot(metrics[1], metrics[0], label=model_name)

# Ligne de référence : la précision du classifieur trivial « tout est fraude »
# (= taux de fraude du test, 20%) — tout ce qui est au-dessus est la valeur ajoutée du modèle
plt.axhline(y=0.20, color="navy", lw=2, linestyle="--", label="Classifieur trivial (tout fraude)")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.title("Courbes Précision/Rappel des 3 modèles")
plt.legend()
plt.show()

#### Étape 4 — Calcul du seuil optimal sous contrainte de rappel ≥ 90%

In [0]:
# Seuil optimal : précision maximale sous contrainte de rappel >= 0,90
import numpy as np

target_recall = 0.90
seuils = {}

for model_name, metrics in precision_recall_curve_metrics.items():
    precision, recall, thresholds = metrics[0], metrics[1], metrics[2]

    # Piège classique : thresholds a exactement 1 élément de MOINS que precision/recall
    # (le dernier point de la courbe, rappel=1, n'a pas de seuil)
    print(f"Model {model_name}: len(precision)={len(precision)}, len(recall)={len(recall)}, len(thresholds)={len(thresholds)}")

    precision_ok = precision[:-1]
    recall_ok = recall[:-1]

    # Points satisfaisant la contrainte de rappel >= 0,90 -> précision maximale -> seuil correspondant
    mask = recall_ok >= target_recall
    best_idx = np.argmax(precision_ok[mask])
    seuil_optimal = thresholds[mask][best_idx]
    precision_at_seuil = precision_ok[mask][best_idx]
    recall_at_seuil = recall_ok[mask][best_idx]

    seuils[model_name] = seuil_optimal
    print(f"  → seuil optimal: {seuil_optimal:.4f} | rappel: {recall_at_seuil:.4f} | précision: {precision_at_seuil:.4f}\n")

In [0]:
# Appliquer le seuil et mesurer : comparaison DIRECTE des probas au seuil
# (on ne passe PAS par .predict(), figé au seuil 0,5)
import pandas as pd
from sklearn.metrics import confusion_matrix

resume = []
for model_name, seuil in seuils.items():
    predictions_seuil = (probas[model_name] >= seuil).astype(int)

    cm = confusion_matrix(y_test_pd["is_fraud"], predictions_seuil)
    vn, fp, fn, vp = cm.ravel()  # ordre [[VN, FP], [FN, VP]]

    nb_alertes = vp + fp   # volume d'investigations que l'équipe anti-fraude devra absorber

    resume.append({
        "modele": model_name,
        "seuil": round(seuil, 4),
        "rappel": round(vp / (vp + fn), 4),
        "precision": round(vp / (vp + fp), 4),
        "alertes_VP_FP": nb_alertes,
        "VP": vp, "FN": fn, "FP": fp, "VN": vn,
    })

resume_df = pd.DataFrame(resume)
display(resume_df)

### B) Analyse des résultats : optimisation du seuil de décision (courbe Précision/Rappel)

**Contexte.** Après la comparaison des trois familles de modèles à seuil par défaut 0,5 (logistic regression, Random Forest, XGBoost — AUC quasi identiques entre 0,856 et 0,864), cette étape optimise le **seuil de décision** : parmi tous les seuils possibles, on retient celui qui **maximise la précision sous contrainte de rappel ≥ 0,90** (la contrainte métier, les faux négatifs coûtant plus cher que les faux positifs). Les modèles entraînés sont rechargés depuis MLflow (`runs:/`), les probabilités calculées sur le test temporel de 19 963 sinistres jamais vus à l'entraînement.

**1. Résultats au seuil optimal.**

| Modèle | Seuil optimal | Rappel | Précision | Alertes (VP+FP) | VP / FN / FP |
| --- | --- | --- | --- | --- | --- |
| logreg_baseline | 0,049 | 0,900 | **0,322** | **11 163** | 3 593 / 399 / 7 570 |
| xgboost_v1 | 0,158 | 0,900 | 0,303 | 11 845 | 3 593 / 399 / 8 252 |
| random_forest_v1 | 0,070 | 0,910 | 0,293 | 12 392 | 3 633 / 359 / 8 759 |

**2. L'objectif de rappel 90% est atteignable — mais son prix est élevé.** Le passage du seuil 0,5 au seuil optimal fait gagner 35 points de rappel à la logreg (0,55 → 0,90), au prix d'une précision qui chute de 0,87 à 0,32 : environ **une vraie fraude pour trois alertes**. Concrètement, 11 000 à 12 400 sinistres sur 19 963 (56 à 62% du volume de test) déclenchent une investigation. C'est la question métier centrale du projet : la capacité d'absorption de l'équipe anti-fraude. Si elle est limitée, le compromis devra être renégocié (par exemple rappel 0,85) en s'appuyant sur les coûts relatifs FN (indemnisation indue) vs FP (investigation manuelle).

**3. La régression logistique s'en sort le mieux à seuil optimisé.** Meilleure précision (0,322) et volume d'alertes le plus faible (11 163), devant XGBoost et Random Forest. Ce résultat, cohérent avec des AUC quasi identiques, confirme que **la capacité de discrimination est saturée par la qualité des features, pas par la complexité du modèle** : le prochain levier n'est pas l'algorithme mais l'ingénierie des features. La passe de tuning (GridSearchCV) affinera les hyperparamètres mais ne devrait pas bouleverser ce classement.

**4. Chaque modèle exige son propre seuil.** Les seuils optimaux diffèrent fortement : 0,049 (logreg), 0,070 (RF), 0,158 (XGBoost). Le seuil XGBoost vaut ~3× celui de la logreg — signature directe du `scale_pos_weight` ≈ 4 qui « gonfle » les probabilités de la classe fraude à dessein. Conséquence pratique : **le seuil est un paramètre du couple (modèle, calibration)**, à recalibrer à chaque réentraînement — il ne se transporte pas d'un modèle à l'autre.

**5. Détail technique : la Random Forest produit des probabilités discrètes.** Sa courbe PR ne compte que 102 points, contre ~20 000 pour les deux autres modèles : 100 arbres qui votent donnent des probabilités par paliers (k/100). La RF ne peut donc pas viser précisément le rappel 0,90 et « atterrit » à 0,91. Les modèles à probabilités continues offrent un réglage plus fin du point de fonctionnement.

**6. Conclusion et suite du plan.** L'optimisation du seuil a livré son enseignement principal : le tri des sinistres est déjà bon (AUC 0,86), c'est le placement de la frontière fraude/honnête qui gouverne l'usage opérationnel. Prochaines étapes : tuning des hyperparamètres par validation croisée temporelle (GridSearchCV + TimeSeriesSplit, folds respectant l'ordre chronologique), puis sélection du modèle final et enregistrement dans le Model Registry avec signature MLflow (le seuil optimal sera embarqué comme paramètre métier du modèle servi, ajustable dans l'application Streamlit).

### C) Point mort économique du choix de seuil — régression logistique

- Objectif : chiffrer en euros le compromis seuil 0,5 → seuil optimal (rappel 0,90) pour transformer le choix du seuil en décision économique défendable

In [0]:
# Point mort économique : relit les montants réels directement dans la table Gold
# (même split temporel 80/20 que partout ailleurs : PERCENT_RANK sur claim_date)
GOLD_TABLE = "workspace.projet_alpha.insurance_claim_fraud_gold"

test_amounts = spark.sql(f"""
    WITH classe AS (SELECT *, PERCENT_RANK() OVER (ORDER BY claim_date ASC) AS progression FROM {GOLD_TABLE})
    SELECT claim_amount, is_fraud
    FROM classe WHERE progression > 0.80
""").toPandas()

# 1) Montant moyen réellement réclamé par un sinistre frauduleux du test
montant_moyen_fraude = test_amounts.loc[test_amounts["is_fraud"] == 1, "claim_amount"].mean()

# 2) Chiffres de la logreg (baseline au seuil 0,5 vs seuil optimal 0,049)
scenarios = {
    "seuil_0.5":            {"VP": 2192, "FP": 318},
    "seuil_optimal_0.049": {"VP": 3593, "FP": 7570},
}

# 3) Point mort : à partir de quel coût d'investigation le seuil bas cesse d'être rentable ?
vp_gagnes = scenarios["seuil_optimal_0.049"]["VP"] - scenarios["seuil_0.5"]["VP"]
fp_supplementaires = scenarios["seuil_optimal_0.049"]["FP"] - scenarios["seuil_0.5"]["FP"]
gain_potentiel = vp_gagnes * montant_moyen_fraude
cout_investigation_max = gain_potentiel / fp_supplementaires

# 4) Lecture métier avec une hypothèse de coût réaliste (à ajuster avec l'entreprise)
COUT_INVESTIGATION = 300  # € par dossier investigué
net_seuil_bas = gain_potentiel - fp_supplementaires * COUT_INVESTIGATION

print(f"Montant moyen d'un sinistre frauduleux (test) : {montant_moyen_fraude:,.0f} €")
print(f"Fraudes en plus détectées (seuil bas vs 0,5)   : {vp_gagnes}")
print(f"Investigations en plus déclenchées             : {fp_supplementaires}")
print(f"Gain brut du seuil bas : {gain_potentiel:,.0f} €")
print(f"\n→ POINT MORT : tant qu'une investigation coûte moins de {cout_investigation_max:,.0f} €,")
print(f"  viser le rappel 0,90 est économiquement rentable.")
print(f"\n→ À {COUT_INVESTIGATION} €/dossier, le seuil bas rapporte {net_seuil_bas:,.0f} € net sur un an de sinistres")

#### D) Point mort économique du choix de seuil — argumentaire coûts

Le choix du seuil n'est pas une affaire de statistique mais d'économie : il se décide en comparant le coût d'une fraude ratée (FN) à celui d'une investigation inutile (FP). Les montants sont calculés sur les données réelles du test.

- **Montant moyen d'un sinistre frauduleux : 7 031 €** (moyenne réelle des `claim_amount` frauduleux du test — les fraudes de ce dataset sont majoritairement des petits sinistres).
- **Comparaison des points de fonctionnement (logreg)** : au seuil 0,5 → 2 192 fraudes détectées pour 318 alertes ; au seuil optimal 0,049 (rappel 0,90) → 3 593 fraudes pour 7 570 alertes.
- **Le prix du rappel 0,90** : +1 401 fraudes détectées (≈ 9,85 M€ de versements induits évités sur un an de sinistres) contre +7 252 investigations supplémentaires.
- **Point mort : ≈ 1 358 €** (9,85 M€ ÷ 7 252). Tant qu'une investigation coûte moins de 1 358 €, viser le rappel 0,90 est économiquement rentable — une enquête réelle coûtant quelques centaines d'euros, la décision est nette.
- **À 300 €/dossier** (hypothèse métier à valider) : gain net ≈ **7,7 M€/an** par rapport au seuil 0,5.
- **Limites de l'exercice** (à citer pour l'honnêteté méthodologique) : on suppose que chaque fraude détectée évite 100% du versement ; une partie des fraudes aurait été interceptée par les contrôles standards ; les FP ont aussi des coûts non monétaires (friction client, risque réglementaire) → d'où l'architecture à deux niveaux (niveau rouge proba élevée = action directe, niveau orange = triage) recommandée pour la production.
- **Conclusion métier** : le seuil devient une **variable d'ajustement de la capacité d'investigation**, recalibrée périodiquement selon les coûts réels et la dérive des données — pas une constante technique.

## 4) Tuning des hyperparamètres et sélection du modèle final pour enregistrement dans Model Registry avec signature MLflow

### Étape 1 — X/y triés par claim_date

In [0]:
# X/y de TRAIN triés par claim_date (prérequis de la validation croisée temporelle)
# Tri côté Spark (orderBy) AVANT toPandas : TimeSeriesSplit découpe par position de ligne,
# la fin du dataframe doit donc être le futur. Séparation X/y APRÈS le tri ; noms _cv pour
# ne pas écraser les originaux ; claim_date jamais dans X (gardée à part pour vérifier le tri).
train_sorted = train_df.orderBy("claim_date").toPandas()

X_train_cv = train_sorted[X_train_pd.columns]   # sélecteur : les 29 features dans le même ordre
y_train_cv = train_sorted[["is_fraud"]]

train_dates = train_sorted[["claim_date"]]

# Vérifications
print("Shape X_train_cv :", X_train_cv.shape, "(attendu : 80037 lignes, 29 colonnes)")
print("Shape y_train_cv :", y_train_cv.shape, "(attendu : 80037 lignes, 1 colonne)")
print("Première claim_date :", train_dates["claim_date"].iloc[0], "(attendu : début 2021)")
print("Dernière claim_date :", train_dates["claim_date"].iloc[-1], "(attendu : 02/01/2025)")
print("claim_date dans X ?", "claim_date" in X_train_cv.columns, "(attendu : False)")

### Étape 2 — GridSearchCV sur les 3 modèles ML

- tuning sur le train
- K-Fold vs TimeSeriesSplit : 
    - k-fold : mélange les lignes aléatoirement => convient pour données indépendantes
    - time series split : découpe par position de ligne

- GridSearch CV: 
    - nb entrainements = n_combinaisons x n_folds

In [0]:
# Construire un pipeline neuf (préprocesseur + classifieur) pour le tuning
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def build_pipeline(classifier_model):
    """Construit le pipeline de preprocessing + classifieur (non entraîné)."""
    numeric_features = ["age", "credit_score", "years_with_company", "previous_claims","annual_income","policy_duration_years", "premium_amount", "coverage_amount", "deductible",
        "witnesses", "injuries", "vehicle_age", "mileage", "vehicle_value",
        "repair_cost", "towing_cost", "medical_cost", "rental_vehicle_cost", "claim_amount","reporting_delay_days", "repair_cost_ratio", "claim_amount_ratio"]

    numeric_transformer = Pipeline(
        steps=[
            ("scaler", StandardScaler())
        ])

    categorical_features = ["accident_severity"]
    categorical_transformer = Pipeline(
        steps=[
            ("encoder", OneHotEncoder(handle_unknown="ignore"))  # ignorer les nouvelles catégories au test
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="passthrough",  # garder les autres colonnes
    )

    pipeline = Pipeline(
        steps=[("preprocessor", preprocessor), ("classifier", classifier_model)])
    
    return pipeline

In [0]:
# Tuning d'un modèle : RandomizedSearchCV avec validation croisée temporelle
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

def tune_fraud_model(X_train_cv, y_train_cv, model_instance,  model_name, grid_params):
    print(f"Tuning model : {model_name}")
    
    pipeline = build_pipeline(model_instance)

    search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=grid_params,
    cv=TimeSeriesSplit(n_splits=3),
    scoring="average_precision",   # métrique indépendante du seuil
    n_jobs=-1,
    random_state=42,
    n_iter=10 # combien de tirages par modèle
    )

    search.fit(X_train_cv, y_train_cv["is_fraud"])

    return search

In [0]:
# Grilles d'hyperparamètres par modèle (préfixe classifier__ pour cibler l'étape du Pipeline)
grid_params_logreg = {
    "classifier__C": [0.01, 0.1, 1, 10],
    "classifier__class_weight": [None, "balanced"]
}

grid_params_rf = {
    "classifier__n_estimators": [100,200,300],
    "classifier__max_depth": [10,20, None],
    "classifier__class_weight": [None, "balanced"]
}

grid_params_xgb = {
        "classifier__n_estimators": [100, 200, 300],
        "classifier__max_depth": [3,5,7],
         "classifier__learning_rate": [0.01, 0.05, 0.1, 0.3]
}

In [0]:
# Instances des 3 modèles (les hyperparamètres seront choisis par la recherche)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models_name = [(LogisticRegression(random_state=42, max_iter=1000), "logreg_baseline_tuned", grid_params_logreg),
              (RandomForestClassifier(random_state=42), "random_forest_tuned", grid_params_rf),
              (XGBClassifier(random_state=42, scale_pos_weight=3.99), "xgboost_tuned", grid_params_xgb)]

In [0]:
# Tuner les 3 modèles et afficher les meilleurs hyperparamètres
grid_searchs={}

for model_instance,model_name,grid_params_model in models_name:
    grid_searchs[model_name] = tune_fraud_model(X_train_cv, y_train_cv, model_instance, model_name, grid_params_model)

for model_name, search in grid_searchs.items():
    print(f"\nMeilleurs hyperparamètres : {search.best_params_}")
    print(f"Meilleur score CV (average_precision, moyenne des 3 folds) : {search.best_score_:.4f}")

#### Analyse des résultats du tuning

### Étape 3 — Calcul du seuil sur le dernier fold de validation

In [0]:
# Récupérer le dernier fold du split temporel pour recalibrer le seuil
from sklearn.model_selection import TimeSeriesSplit

tss = TimeSeriesSplit(n_splits=3)
toutes_les_paires = list(tss.split(X_train_cv))

for i, (train_idx, val_idx) in enumerate(toutes_les_paires, start=1):
    print(f"Fold {i} : train = {len(train_idx):,} lignes | validation = {len(val_idx):,} lignes")

# La cible : la dernière paire (le futur chronologique du train)
dernier_train_idx, dernier_val_idx = toutes_les_paires[-1]
print(f"\nDernier fold → train_idx va de {dernier_train_idx.min()} à {dernier_train_idx.max()} (indices positions)")
print(f"Dernier fold → val_idx   va de {dernier_val_idx.min()} à {dernier_val_idx.max()} (indices positions)")

In [0]:
# Pour chaque modèle tuné : réentraîner un "jumeau" (mêmes hyperparamètres, entraînement
# effacé via clone()) sur le dernier fold, puis calculer sa courbe PR sur la validation
from sklearn.base import clone
from sklearn.metrics import precision_recall_curve
import numpy as np

jumeaux={}
metrics_models={}
for model_name, search in grid_searchs.items():
    metrics_model=[]
    jumeau = clone(search.best_estimator_) # copie les hyperparamètres, EFFACE l'entraînement
    jumeaux[model_name] = jumeau
    # Tranches du fold — MÊMES indices pour X et y, dans chaque rôle
    X_train_fold = X_train_cv.iloc[dernier_train_idx]              # 60 028 lignes : le passé
    y_train_fold = y_train_cv["is_fraud"].iloc[dernier_train_idx]  # leurs étiquettes
    X_val_fold = X_train_cv.iloc[dernier_val_idx]                   # 20 009 lignes : le pseudo-futur
    y_val_fold = y_train_cv["is_fraud"].iloc[dernier_val_idx]      # leurs étiquettes

    jumeau.fit(X_train_fold, y_train_fold)
    probas_val_tuned = jumeau.predict_proba(X_val_fold)[:, 1]
    precision, recall, thresholds = precision_recall_curve(y_val_fold, probas_val_tuned) 
    metrics_model.append(precision)
    metrics_model.append(recall)
    metrics_model.append(thresholds)
    metrics_models[model_name] = metrics_model

print("Jumeaux calibrés :", list(jumeaux.keys()))

In [0]:
# Seuil optimal de chaque modèle tuné, calibré sur la validation (dernier fold)
import numpy as np

target_recall = 0.90
seuils_tuned={}

for model_name, metrics in metrics_models.items():
    precision, recall, thresholds = metrics[0], metrics[1], metrics[2]

    # Piège classique : thresholds a exactement 1 élément de MOINS que precision/recall
    print(f"Model {model_name}: len(precision)={len(precision)}, len(recall)={len(recall)}, len(thresholds)={len(thresholds)}")

    precision_ok = precision[:-1]
    recall_ok = recall[:-1]

    mask = recall_ok >= target_recall
    best_idx = np.argmax(precision_ok[mask])
    seuil_optimal = thresholds[mask][best_idx]
    precision_at_seuil = precision_ok[mask][best_idx]
    recall_at_seuil = recall_ok[mask][best_idx]

    seuils_tuned[model_name] = seuil_optimal
    print(f"  → seuil optimal: {seuil_optimal:.4f} | rappel: {recall_at_seuil:.4f} | précision: {precision_at_seuil:.4f}\n")

##### Tableau comparatif des seuils avec la version 1

| Modèle tuné | Seuil calibré (validation) | Rappel | Précision | vs v1 |
| --- | --- | --- | --- | --- |
| logreg (balanced, C=0,01) | 0,2224 | 0,900 | 0,332 | v1 : 0,049 → ×4,5 |
| RF (300 arbres, depth 10, None) | 0,0744 | 0,900 | 0,324 | v1 : 0,070 |
| XGB (scale_pos_weight 3,99) | 0,1754 | 0,900 | 0,328 | v1 : 0,158 |

### Étape 4 — Évaluation sur le test

In [0]:
# Probabilités sur le test avec les best_estimator_ (entraînés sur tout le train)
probas_test_tuned={}
for model_name, search in grid_searchs.items():
    print(f"Model {model_name} :")
    print(f"  - best params: {search.best_params_}")
    print(f"  - best score: {search.best_score_}")
    print(f"  - seuil optimal: {seuils_tuned[model_name]:.4f}")
    model_tuned= search.best_estimator_
    probas_test = model_tuned.predict_proba(X_test_pd)[:, 1]
    probas_test_tuned[model_name] = probas_test

In [0]:
# Appliquer le seuil calibré et mesurer sur le test (comparaison DIRECTE des probas au seuil)
import pandas as pd
from sklearn.metrics import confusion_matrix

resume_tuned = []
for model_name, seuil in seuils_tuned.items():
    predictions_seuil_tuned = (probas_test_tuned[model_name] >= seuil).astype(int)

    cm = confusion_matrix(y_test_pd["is_fraud"], predictions_seuil_tuned)
    vn, fp, fn, vp = cm.ravel()  # ordre [[VN, FP], [FN, VP]]

    nb_alertes = vp + fp   # volume d'investigations que l'équipe anti-fraude devra absorber

    resume_tuned.append({
        "modele": model_name,
        "seuil": round(seuil, 4),
        "rappel": round(vp / (vp + fn), 4),
        "precision": round(vp / (vp + fp), 4),
        "alertes_VP_FP": nb_alertes,
        "VP": vp, "FN": fn, "FP": fp, "VN": vn,
    })

resume_tuned_df = pd.DataFrame(resume_tuned)
display(resume_tuned_df)

#### Analyse des métriques du tuning des 3 modèles ML

**Contexte et méthode.** Le tuning utilise `RandomizedSearchCV` (10 tirages par modèle, `random_state=42` pour la reproductibilité) avec validation croisée **temporelle** (`TimeSeriesSplit` à 3 folds sur le train trié par `claim_date`) et le scoring `average_precision` — une métrique indépendante du seuil, qui mesure la qualité du tri. Le seuil métier est ensuite **calibré sur le dernier fold de validation** (le futur chronologique du train, via un modèle jumeau non entraîné par `clone()`), et les modèles sont évalués **une seule fois sur le test temporel** jamais utilisé pour calibrer. Question posée : le tuning améliore-t-il le point de fonctionnement métier (précision à rappel ≥ 0,90) par rapport aux modèles v1 ?

**1. Résultats du tuning et évaluation sur le test.**

| Modèle | Meilleurs hyperparamètres | Score CV (AP) | Seuil calibré | Rappel (test) | Précision (test) | Alertes (VP+FP) |
| --- | --- | --- | --- | --- | --- | --- |
| logreg tunée | balanced, C=0,01 | 0,759 | 0,2224 | 0,887 | 0,343 | 10 334 |
| RF tunée | 300 arbres, depth 10, None | 0,761 | 0,0744 | 0,890 | 0,342 | 10 391 |
| **XGB tuné** | 100 arbres, depth 5, lr 0,05 | 0,763 | 0,1754 | 0,880 | **0,367** | **9 566** |

**2. Le tuning améliore les trois modèles au point métier.** Par rapport à la v1 au même critère : précision 0,322 → 0,343 (logreg), 0,293 → 0,342 (RF), 0,303 → 0,367 (XGB) ; volumes d'alertes en baisse de 8 à 19 %. La Random Forest profite le plus du passage de 100 arbres libres à 300 arbres bornés (`max_depth=10`) : moins de surapprentissage, probabilités plus fines.

**3. Changement de leader : le XGBoost tuné.** En v1, la régression logistique gagnait (précision 0,322) ; après tuning, le XGBoost prend la tête : meilleure précision (0,367) et surtout le volume d'alertes le plus faible (9 566, soit **une vraie fraude pour 2,7 alertes** contre 3,1 en v1 — environ 1 568 investigations évitées par an pour un rappel équivalent). Le gradient boosting avec `scale_pos_weight=3,99` et une configuration volontairement prudente est la famille qui bénéficie le plus d'hyperparamètres bien choisis.

**4. Un verdict nuancé — et important à comprendre.** Les scores CV (`average_precision` 0,759 à 0,763) restent quasi identiques entre familles, comme les AUC en v1 : la capacité de **tri global** des sinistres reste saturée par la qualité des features. Ce que le tuning a réellement amélioré, c'est le **point de fonctionnement** : la distribution des probabilités autour de la frontière de décision est mieux placée. Le classement des sinistres était déjà bon (AUC 0,86) ; le tuning affine le placement, il ne réinvente pas le tri.

**5. Les enseignements des hyperparamètres retenus.**
* **logreg** : `class_weight=balanced` + forte régularisation (C=0,01) → probabilités gonflées côté fraude → seuil multiplié par 4,5 (0,049 → 0,2224). Confirmation empirique de la thèse « le seuil est un paramètre du couple (modèle, calibration) ».
* **RF** : `class_weight=None` gagne — la pondération « balanced » de la v1 ne servait pas la précision moyenne (mystère v1 élucidé). Et ses 300 arbres bornés produisent des probabilités quasi continues (20 009 points de courbe PR contre 102 en v1) : le réglage fin du seuil devient possible.
* **XGB** : configuration prudente (peu d'arbres, profondeur modérée, petit pas d'apprentissage) — sur des données temporelles, l'anti-surapprentissage gagne.

**6. L'écart de généralisation assumé.** Les rappels sur le test (0,88 à 0,89) restent légèrement sous la cible de 0,90 calibrée en validation : le seuil ajusté sur le pseudo-futur du train ne tombe pas exactement à 0,90 sur le vrai futur. C'est précisément la mesure honnête que la « version propre » du seuil visait — c'est le prix (2 points de rappel) de la rigueur méthodologique, et un phénomène à documenter comme attendu.

**7. Décision.** Le **XGBoost tuné est retenu comme modèle final** : meilleure précision (0,367) et volume d'alertes le plus faible (9 566/an), au prix d'un rappel de 0,88 — deux points sous la cible. Arbitrage documenté : le seuil restant un paramètre métier ajustable au déploiement, l'entreprise pourra le baisser pour revenir à 0,90 en payant le prix correspondant en précision. Les trois modèles tunés sont journalisés dans MLflow (avec la table de recherche `cv_results_` en artefact), et le XGBoost gagnant est enregistré dans le Model Registry avec signature et seuil métier en paramètre — prêt pour l'application Streamlit.

In [0]:
# Coût d'investigation des alertes du XGBoost tuné (modèle final retenu)
# Question métier : combien coûte le traitement du volume d'alertes (VP + FP) au seuil
# métier calibré, et combien rapporte-t-il ? Ce n'est plus un point mort entre deux seuils :
# c'est le budget annuel de l'équipe d'investigation face aux versements induits évités.
# Prérequis : seuils_tuned et probas_test_tuned calculés plus haut.

MODEL_NAME = "xgboost_tuned"
COUT_INVESTIGATION = 300  # € par dossier investigué (hypothèse métier à valider)

# 1) Montant moyen réel d'un sinistre frauduleux du test (relu depuis la Gold)
GOLD_TABLE = "workspace.projet_alpha.insurance_claim_fraud_gold"

test_amounts = spark.sql(f"""
    WITH classe AS (SELECT *, PERCENT_RANK() OVER (ORDER BY claim_date ASC) AS progression FROM {GOLD_TABLE})
    SELECT claim_amount, is_fraud
    FROM classe WHERE progression > 0.80
""").toPandas()

montant_moyen_fraude = test_amounts.loc[test_amounts["is_fraud"] == 1, "claim_amount"].mean()

# 2) Point de fonctionnement au seuil métier : volume d'alertes calculé sur SES probas de test
from sklearn.metrics import confusion_matrix

seuil_metier = seuils_tuned[MODEL_NAME]          # calibré sur le dernier fold de validation
probas_xgb = probas_test_tuned[MODEL_NAME]       # probas du best_estimator_ sur le test

predictions = (probas_xgb >= seuil_metier).astype(int)   # comparaison directe au seuil

cm = confusion_matrix(y_test_pd["is_fraud"], predictions)
vn, fp, fn, vp = cm.ravel()

nb_alertes = vp + fp   # le volume que l'équipe anti-fraude devra absorber (vrais + faux positifs)

# 3) Le budget d'investigation et ce qu'il rapporte
cout_investigation_total = nb_alertes * COUT_INVESTIGATION
gain_brut = vp * montant_moyen_fraude           # versements induits évités sur les fraudes détectées
benefice_net = gain_brut - cout_investigation_total
roi = gain_brut / cout_investigation_total       # € évités par € investi en investigation

# 4) Référence sans modèle : investiguer TOUS les sinistres pour le même objectif
nb_sinistres = len(y_test_pd)
cout_sans_modele = nb_sinistres * COUT_INVESTIGATION

print(f"Montant moyen d'un sinistre frauduleux (test) : {montant_moyen_fraude:,.0f} €")
print(f"Alertes du XGB tuné au seuil métier ({seuil_metier:.4f}) : {nb_alertes:,} (VP={vp:,} + FP={fp:,})")
print(f"\n→ BUDGET d'investigation : {nb_alertes:,} alertes × {COUT_INVESTIGATION} € = {cout_investigation_total:,.0f} €/an")
print(f"→ Gain brut (versements induits évités) : {vp:,} fraudes × {montant_moyen_fraude:,.0f} € = {gain_brut:,.0f} €/an")
print(f"→ Bénéfice net : {benefice_net:,.0f} €/an | ROI : {roi:.1f} € évités par € d'investigation")
print(f"\n→ Référence sans modèle (investiguer les {nb_sinistres:,} sinistres) : {cout_sans_modele:,.0f} €/an")
print(f"  Le modèle divise le coût d'investigation par {cout_sans_modele/cout_investigation_total:.1f}")
print(f"  tout en attrapant {vp/(vp+fn)*100:.0f} % des fraudes réelles")

### Étape 5 — MLflow et Registry du meilleur modèle tuné

In [0]:
# Étape 5a — Journaliser les 3 modèles TUNÉS dans MLflow (runs à côté des v1, même expérience)
# On ne réentraîne rien : les best_estimator_ existent déjà ; il ne reste qu'à les tracer
# (hyperparamètres gagnants, métriques, table de recherche, modèle avec signature).

import mlflow
import pandas as pd

mlflow.set_experiment(MLFLOW_EXPERIMENT)

runs_tuned_ids = {}   # run_id par modèle — servira au Registry

# Colonnes entières → déclarées en double (float64) dans la signature : la signature
# tolère alors ints ET floats à l'inférence (fix du warning « integer columns »)
INT_COLS = X_test_pd.select_dtypes(include=["int"]).columns

for model_name, search in grid_searchs.items():
    with mlflow.start_run(run_name=model_name) as run:

        # 1) Paramètres : hyperparamètres gagnants + seuil métier calibré
        mlflow.log_params(search.best_params_)
        mlflow.log_param("seuil_metier", round(seuils_tuned[model_name], 4))

        # 2) Métriques : score CV de la recherche + métriques TEST au seuil calibré
        mlflow.log_metric("cv_average_precision", search.best_score_)
        ligne = resume_tuned_df[resume_tuned_df["modele"] == model_name].iloc[0]
        mlflow.log_metric("rappel_test_au_seuil", ligne["rappel"])
        mlflow.log_metric("precision_test_au_seuil", ligne["precision"])
        mlflow.log_metric("alertes_test_VP_FP", ligne["alertes_VP_FP"])

        # 3) Artefact : la table de recherche complète (10 tirages × 3 folds)
        #    = la trace de COMMENT les hyperparamètres ont été choisis
        cv_csv = f"cv_results_{model_name}.csv"
        pd.DataFrame(search.cv_results_).to_csv(cv_csv, index=False)
        mlflow.log_artifact(cv_csv)

        # 4) Le modèle : best_estimator_ (pipeline complet) AVEC input_example
        #    → MLflow en déduit la SIGNATURE (colonnes + types attendus)
        exemple = X_test_pd.head(5).copy()
        exemple[INT_COLS] = exemple[INT_COLS].astype("float64")
        mlflow.sklearn.log_model(
            sk_model=search.best_estimator_,
            artifact_path="pipeline",              # même chemin d'artefact que les v1
            input_example=exemple,                # 5 lignes réelles, types tolérants
        )

        runs_tuned_ids[model_name] = run.info.run_id
        print(f"Run '{model_name}' loggué : {run.info.run_id}")

In [0]:
# Étape 5b — Enregistrer le GAGNANT dans le Model Registry (Unity Catalog)
# MLflow = journal d'expériences (les 3 tunés y sont tracés) ;
# Registry = catalogue de déploiement (un seul modèle part en production).
# Le gagnant désigné par l'analyse comparative : xgboost_tuned.
# NB : chaque exécution de register_model crée une NOUVELLE version du modèle.

import mlflow
from mlflow import MlflowClient

MODEL_REGISTRY_NAME = "workspace.projet_alpha.insurance_fraud_detector"   # format Unity Catalog : catalog.schema.modèle

run_id_gagnant = runs_tuned_ids["xgboost_tuned"]

result = mlflow.register_model(
    model_uri=f"runs:/{run_id_gagnant}/pipeline",   # pointe vers l'artefact « pipeline » du run gagnant
    name=MODEL_REGISTRY_NAME,
)
print(f"Modèle enregistré : {MODEL_REGISTRY_NAME} — version {result.version}")

client = MlflowClient()

# Alias « champion » : l'application Streamlit chargera TOUJOURS la version ainsi marquée,
# via l'URI models:/workspace.projet_alpha.insurance_fraud_detector@champion
# → déployer une nouvelle version se fait sans toucher au code de l'app.
client.set_registered_model_alias(MODEL_REGISTRY_NAME, "champion", result.version)

# Le seuil métier accroché au modèle (affichage + réglage côté application).
# Le nom de la méthode de tag varie selon la version de MLflow : on essaie les candidats.
# Dans tous les cas, le seuil vit déjà en PARAMÈTRE du run MLflow.
seuil_value = str(round(seuils_tuned["xgboost_tuned"], 4))
tag_applique = False
for methode in ("set_registered_model_version_tag", "set_model_version_tag"):
    fn = getattr(client, methode, None)
    if fn is not None:
        fn(MODEL_REGISTRY_NAME, result.version, "seuil_metier", seuil_value)
        print(f"Tag 'seuil_metier' appliqué à la version {result.version} (via {methode})")
        tag_applique = True
        break
if not tag_applique:
    try:
        client.set_registered_model_tag(MODEL_REGISTRY_NAME, "seuil_metier", seuil_value)
        print("Tag 'seuil_metier' appliqué au niveau du modèle enregistré")
    except AttributeError:
        print("Info : 'seuil_metier' reste tracé en paramètre du run MLflow")

print("Alias 'champion' appliqué — prêt pour l'application Streamlit.")

In [0]:
# Diagnostic — pourquoi le warning « integer columns » persiste-t-il ?
# On inspecte les dtypes RÉELS de X_test_pd, ce que INT_COLS a capturé,
# et la signature réellement enregistrée sur le champion (alias).

print("Répartition des dtypes dans X_test_pd :")
print(X_test_pd.dtypes.value_counts())

print(f"\nColonnes capturées par INT_COLS ({len(INT_COLS)}) : {list(INT_COLS)}")

exemple = X_test_pd.head(5).copy()
exemple[INT_COLS] = exemple[INT_COLS].astype("float64")

print("\nDtypes de l'exemple APRÈS cast :")
print(exemple.dtypes.value_counts())

restants = [c for c in exemple.columns if pd.api.types.is_integer_dtype(exemple[c])]
print(f"\nColonnes ENTIÈRES restantes dans l'exemple : {restants}")

# Vérification finale : la signature RÉELLEMENT enregistrée sur le champion (alias)
import mlflow
champion = mlflow.pyfunc.load_model("models:/workspace.projet_alpha.insurance_fraud_detector@champion")
sig = champion.metadata.signature
print("\n--- SIGNATURE DU CHAMPION (enregistrée) ---")
print("Entrée :")
print(sig.inputs)
print("\nSortie :")
print(sig.outputs)